In [ ]:
from pyserini.search import get_topics, get_qrels

from tqdm import tqdm

import transformers
import torch

import json

model_id = "autodl-tmp/LLM-Research/Meta-Llama-3-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",
)

topics = get_topics('dl19-passage')
qrels = get_qrels('dl19-passage')


queries=[]
for qid in tqdm(topics):
    if qid in qrels.keys():
        query = [qid,topics[qid]['title']]
        queries.append(query)

prompts=queries
lenprompts=len(prompts)

for i in range(lenprompts):
    print(i)
    t1=time.time()
    prompt_text=prompts[i][1]
    
    messages = [
        {"role": "system", "content": 'Please write a passage to answer the question.'},
        {"role": "user", "content": prompt_text},
    ]

    prompt = pipeline.tokenizer.apply_chat_template(
		    messages, 
		    tokenize=False, 
		    add_generation_prompt=True
    )

    terminators = [
        pipeline.tokenizer.eos_token_id,
        pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]


    for j in range(5):
        outputs = pipeline(
            prompt,
            max_new_tokens=128,
            eos_token_id=terminators,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
        )
        hypothesis_doc=outputs[0]["generated_text"][len(prompt):].replace('\n','')
        queries[i].append(hypothesis_doc)

    with open('GOLFer_dl19/hypothesis_documents_dl19_5', 'w') as file:
        json.dump(queries, file)
